# External connectivity: a local MCP calculator

Copyright (c) Patrick Hall (jphall@gwu.edu), 2026-27. Released under the MIT License; see [SOFTWARE_LICENSE.txt](../../SOFTWARE_LICENSE.txt).

Discover a tool on a **separate local server process**, expose its contract to the model, dispatch the requested call through MCP, and return the result for a final answer. The model calculates revenue growth from $1.2 million to $1.5 million using the discovered tool. The notebook is the host; `ClientSession` is the client; `calculator_server.py` is the server. The model API and local MCP connection are separate connections.

From the repository root: `uv sync --locked`, then `uv run jupyter lab lecture/lec_6/`. Run cells in order. This notebook starts and stops its own server subprocess over stdio; no port, hosted MCP service, or second terminal is required. `uv run python lecture/lec_6/calculator_server.py` starts the same server manually for another stdio-capable client; it waits for protocol messages, not typed arithmetic. Do not start that manual process for this notebook.

The server needs no API credential, network access, or database. Only the host calls the direct OpenAI API. The example uses the locked **MCP Python SDK v1 API**; it is intentionally version constrained below v2.

## 1. Configure the host
Use the same project-root credential loading and model setting as the Text2SQL notebook.

In [1]:
# 1. Configure the direct OpenAI client
import os
import sys
import json
from pathlib import Path
from getpass import getpass
from time import monotonic
from dotenv import load_dotenv
from openai import OpenAI

# Find the project by its configuration, even if .env has not been created yet.
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'pyproject.toml').is_file())
lecture_dir = project_root / 'lecture' / 'lec_6'
load_dotenv(project_root / '.env')
api_key = os.getenv('OPENAI_API_KEY') or getpass('OpenAI API key: ')
client = OpenAI(api_key=api_key, timeout=45, max_retries=1)
MODEL = os.getenv('OPENAI_TEXT_MODEL', 'gpt-5-mini')
print('Model:', MODEL)  # Never print credentials.

Model: gpt-5-mini


## 2. Inspect the separate server code

The server exports one typed arithmetic operation. It validates finite revenues and a positive denominator, computes the percentage, and returns structured data. It does not evaluate arbitrary Python expressions. stdout is reserved for MCP messages; logs belong on stderr.

Sources: [MCP Python SDK v1](https://github.com/modelcontextprotocol/python-sdk/tree/v1.x), [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture).

In [2]:
# 2. Display the exact server implementation used by the subprocess
server_path = lecture_dir / 'calculator_server.py'
print(server_path.read_text())

"""Separate local MCP calculator server. Copyright Patrick Hall, 2026-27. MIT License."""
import math
import os
from mcp.server.fastmcp import FastMCP

# The calculator has no need to inherit the host's API credential.
os.environ.pop('OPENAI_API_KEY', None)
mcp = FastMCP('Lecture 6 calculator')


@mcp.tool()
def revenue_growth(previous: float, current: float) -> dict:
    """Return percentage revenue growth: 100 * (current - previous) / previous.

    Inputs must be finite, nonnegative revenues in the same currency and scale;
    previous must be positive. This performs arithmetic only, with no side effects.
    """
    if not math.isfinite(previous) or not math.isfinite(current) or previous <= 0 or current < 0:
        raise ValueError('Use finite revenues, previous > 0 and current >= 0.')
    return {'previous': previous, 'current': current,
            'growth_percent': 100 * (current - previous) / previous}


if __name__ == '__main__':
    # stdout carries MCP JSON-RPC. Send any di

## 3. Discover, request, execute, and answer

`stdio_client` launches the server using this kernel's Python interpreter and cleans it up when its context closes. After initialization, `list_tools()` discovers the name, description, and argument schema. The host sends that contract to the model. The returned request is displayed **before execution**, checked against the approved tool name, validated locally, and sent with `call_tool`.

For this worked example, `tool_choice='required'` requires one tool request, and `parallel_tool_calls=False` keeps the handoff visible. The final call receives the observation and no tools. This is a fixed two-request workflow, not a general-purpose autonomous agent. Notebook top-level `await` works with Jupyter's existing event loop; do not wrap it in `asyncio.run()`.

Source: [OpenAI function calling](https://developers.openai.com/api/docs/guides/function-calling).

In [3]:
# 3. Launch the server, discover its contract, and execute one live model request
import asyncio
import math
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from pydantic import BaseModel, ConfigDict, Field

class GrowthArgs(BaseModel):
    model_config = ConfigDict(extra='forbid', strict=True, allow_inf_nan=False)
    previous: float = Field(gt=0)
    current: float = Field(ge=0)

async def calculator_demo():
    # Explicitly pass a small environment: the server never receives the API key.
    env = {k:v for k,v in os.environ.items() if k in
           {'PATH','SYSTEMROOT','WINDIR','TEMP','TMP','LANG'}}
    parameters = StdioServerParameters(command=sys.executable, args=[str(server_path)], env=env)
    async with stdio_client(parameters) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            discovered = await session.list_tools()
            tool = next(t for t in discovered.tools if t.name == 'revenue_growth')
            print('DISCOVERED:', tool.model_dump_json(indent=2))
            # Only expose the explicitly approved discovered tool, not every server capability.
            schema = dict(tool.inputSchema)
            schema['additionalProperties'] = False
            contract = {'type':'function', 'name':tool.name, 'description':tool.description,
                        'parameters':schema, 'strict':True}
            question = 'Use the calculator to find percentage revenue growth from $1.2 million to $1.5 million.'
            request = await asyncio.to_thread(
                client.responses.create, model=MODEL, input=question, tools=[contract],
                tool_choice='required', parallel_tool_calls=False, max_output_tokens=1800,
                reasoning={'effort':'low'}, store=False)
            if request.status != 'completed':
                raise RuntimeError(f'Incomplete model request: {request.status}')
            calls = [x for x in request.output if x.type=='function_call']
            assert len(calls)==1 and calls[0].name==tool.name
            call = calls[0]
            print('MODEL REQUEST:', call.name, call.arguments)
            args = GrowthArgs.model_validate_json(call.arguments)
            result = await session.call_tool(call.name, arguments=args.model_dump())
            print('MCP RESULT:', result.model_dump_json(indent=2))
            if result.isError:
                raise RuntimeError('Calculator tool failed; do not request an unsupported final answer.')
            # Check both modern structured output and the v1 text-content representation.
            payload = result.structuredContent
            if payload is None:
                payload = json.loads(next(c.text for c in result.content if c.type=='text'))
            assert math.isclose(payload['growth_percent'], 25.0)
            assert math.isclose((args.current-args.previous)/args.previous, 0.25)
            final = await asyncio.to_thread(
                client.responses.create, model=MODEL,
                instructions='Answer only from the calculator result. State the percentage and denominator.',
                input=[{'role':'user','content':question}, *request.output,
                       {'type':'function_call_output','call_id':call.call_id,
                        'output':json.dumps(payload)}],
                max_output_tokens=1200, reasoning={'effort':'low'}, store=False)
            if final.status != 'completed' or not final.output_text:
                raise RuntimeError('The model did not produce a completed final answer.')
            print('FINAL ANSWER:', final.output_text)
            print('MODEL USAGE:', request.usage.model_dump(), final.usage.model_dump())
            # A server-side error is observable protocol data, not a successful calculation.
            invalid = await session.call_tool(tool.name, arguments={'previous':0,'current':1.5})
            assert invalid.isError
            print('ZERO-DENOMINATOR CHECK: rejected by the server')
            return {'growth_percent':payload['growth_percent'], 'answer':final.output_text,
                    'tool_name':tool.name}

# Bound protocol/model work as well as individual HTTP requests; context managers clean up.
mcp_demo = await asyncio.wait_for(calculator_demo(), timeout=120)
assert mcp_demo['growth_percent'] == 25.0

DISCOVERED: {
  "name": "revenue_growth",
  "title": null,
  "description": "Return percentage revenue growth: 100 * (current - previous) / previous.\n\nInputs must be finite, nonnegative revenues in the same currency and scale;\nprevious must be positive. This performs arithmetic only, with no side effects.\n",
  "inputSchema": {
    "properties": {
      "previous": {
        "title": "Previous",
        "type": "number"
      },
      "current": {
        "title": "Current",
        "type": "number"
      }
    },
    "required": [
      "previous",
      "current"
    ],
    "title": "revenue_growthArguments",
    "type": "object"
  },
  "outputSchema": null,
  "icons": null,
  "annotations": null,
  "meta": null,
  "execution": null
}


MODEL REQUEST: revenue_growth {"previous":1200000,"current":1500000}
MCP RESULT: {
  "meta": null,
  "content": [
    {
      "type": "text",
      "text": "{\n  \"previous\": 1200000.0,\n  \"current\": 1500000.0,\n  \"growth_percent\": 25.0\n}",
      "annotations": null,
      "meta": null
    }
  ],
  "structuredContent": null,
  "isError": false
}


FINAL ANSWER: 25% growth (denominator: $1,200,000).
MODEL USAGE: {'input_tokens': 195, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 119, 'output_tokens_details': {'reasoning_tokens': 64}, 'total_tokens': 314} {'input_tokens': 201, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 18, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 219}
ZERO-DENOMINATOR CHECK: rejected by the server


## 4. Interpret the handoffs

The arithmetic result is **25%**: `(1.5 − 1.2) / 1.2 × 100`. Input revenues must use the same units. The model requested a call; the host validated and routed it; the server calculated; the host returned the observation; the model explained the result.

MCP standardizes discovery and invocation, not business authority or answer correctness. The host still approves tool names, limits calls, validates inputs, and controls credentials. A local stdio server is a trusted program running with the host's OS privileges; production isolation and remote-server authentication require additional design. This notebook does not demonstrate remote HTTP hosting.

Sources: [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture), [official MCP Python SDK v1 documentation](https://github.com/modelcontextprotocol/python-sdk/tree/v1.x), [OpenAI function calling](https://developers.openai.com/api/docs/guides/function-calling).